# POLLICINO — UDA 2: Compressione come predizione
Notebook comune per dimostrazione, recupero e discussione in classe. Usa solo la libreria standard Python.

**Filo logico:** ridondanza → codice → probabilità → contesto → bits-per-byte.


In [ ]:
def rle_encode(data: bytes) -> bytes:
    if not data: return b''
    out=bytearray(); value=data[0]; count=1
    for b in data[1:]:
        if b == value and count < 255: count += 1
        else: out.extend((count, value)); value=b; count=1
    out.extend((count, value))
    return bytes(out)

for sample in [b'A'*100, bytes(range(100))]:
    enc=rle_encode(sample)
    print(len(sample), len(enc), len(enc)/len(sample))


In [ ]:
from collections import Counter
import math
def entropy_bpb(data: bytes) -> float:
    if not data: return 0.0
    counts=Counter(data); n=len(data)
    return -sum((c/n)*math.log2(c/n) for c in counts.values())
print(entropy_bpb(b'A'*1000), entropy_bpb(b'AB'*500))


In [ ]:
def bigram_bpb(train: bytes, test: bytes, alpha=0.1):
    table={}; global_counts=[0]*256
    for b in train: global_counts[b]+=1
    for i in range(1,len(train)): table.setdefault(train[i-1:i],[0]*256)[train[i]] += 1
    bits=0.0
    for i in range(1,len(test)):
        counts=table.get(test[i-1:i], global_counts)
        p=(counts[test[i]]+alpha)/(sum(counts)+256*alpha)
        bits += -math.log2(p)
    return bits/(len(test)-1)
train=b'AB'*1000; test=b'AB'*200
print('uniform:',8.0,'bigram:',bigram_bpb(train,test))


## Domande finali
1. Perché RLE può espandere alcuni dati?
2. Perché il codebook di Huffman va contato?
3. Qual è la differenza tra entropia e cross-entropy?
4. Perché il test set non va usato per addestrare?
5. In che senso questa UDA anticipa il training di un language model?
